In [9]:
import pandas as pd
import numpy as np

# Dataset

data = {
'Weather':
['Sunny','Sunny','Cloudy','Rainy','Rainy','Rainy','Cloudy','Sunny','Sunny','Rainy','Sunny','Cloudy','Cloudy','Rainy'],
'Temperature':
['Hot','Hot','Hot','Mild','Cool','Cool','Cool','Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
'Soil Moisture':
['Dry','Dry','Dry','Dry','Moist','Moist','Moist','Dry','Moist','Moist','Moist','Dry','Moist','Dry'],
'Wind':['Weak','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Strong'],
'Irrigate':
 ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','Yes','Yes','Yes','Yes','No']
}
df = pd.DataFrame(data)
print(df)

   Weather Temperature Soil Moisture    Wind Irrigate
0    Sunny         Hot           Dry    Weak       No
1    Sunny         Hot           Dry  Strong       No
2   Cloudy         Hot           Dry    Weak      Yes
3    Rainy        Mild           Dry    Weak      Yes
4    Rainy        Cool         Moist    Weak      Yes
5    Rainy        Cool         Moist  Strong       No
6   Cloudy        Cool         Moist  Strong      Yes
7    Sunny        Mild           Dry    Weak       No
8    Sunny        Cool         Moist    Weak      Yes
9    Rainy        Mild         Moist    Weak      Yes
10   Sunny        Mild         Moist  Strong      Yes
11  Cloudy        Mild           Dry  Strong      Yes
12  Cloudy         Hot         Moist    Weak      Yes
13   Rainy        Mild           Dry  Strong       No


In [11]:
# Step 1: Entropy

def entropy(target):
  values,counts=np.unique(target,return_counts=True)
  prob=counts/counts.sum()
  return -np.sum(prob*np.log2(prob))

# STEP 2: INFROMATION GAIN
def information_gain(data,feature,target_name):
  total_entropy= entropy(data[target_name])

  values,counts=np.unique(data[feature],return_counts=True)

  weighted_entropy=0
  for i in range(len(values)):
    subset=data[data[feature]==values[i]]
    weighted_entropy+=(counts[i]/np.sum(counts))*entropy(subset[target_name])
    return total_entropy-weighted_entropy

# step 3 :ID3 ALGOROTHM
def id3(data,features,target_name,depth=0):
  indent=""*depth

  # if all target values are same== return leaf
  if len(np.unique(data[target_name]))==1:
    return np.unique(data[target_name])[0]

  # if no feature left== retun majority classes
  if len(features)==0:
    return data[target_name].mode()[0]

  # compute ig for all features
  gains=[]
  print(f"\n{indent}Evaluating features at depth{depth}:")


  for feature in features:
    gain=information_gain(data,feature,target_name)
    gains.append(gain)
    print(f"{indent} IG({feature}) = {gain:.4f}")

  # select best feature
  best_feature=features[np.argmax(gains)]
  print(f"{indent}Best Feature == {best_feature}")

  tree={best_feature:{}}

  # split dataset
  for value in np.unique(data[best_feature]):
    print(f"{indent} Splitting {best_feature} = {value}")
    subset=data[data[best_feature]==value]

    if subset.shape[0]==0:
      tree[best_feature][value]=data[target_name].mode()[0]
    else:
      remaining_features=[f for f in features if f!= best_feature]
      subtree=id3(subset,remaining_features,target_name,depth+1)
      tree[best_feature][value]=subtree
  return tree

# Step 4 Print tree function
def print_tree(tree,indent=""):
  if not isinstance(tree,dict):
    print(indent+"--->",tree)
    return
  for feature,branches in tree.items():
    for value,subtree in branches.items():
      print(f"{indent}{feature}={value}:")
      print_tree(subtree,indent+" ")

# Build tree
features=list(df.columns[:-1])

tree_model=id3(df,features,"Irrigate")

# display tree
print("\n Final Decision Tree:\n")
print_tree(tree_model)





Evaluating features at depth0:
 IG(Weather) = 0.9403
 IG(Temperature) = 0.7085
 IG(Soil Moisture) = 0.4477
 IG(Wind) = 0.5117
Best Feature == Weather
 Splitting Weather = Cloudy
 Splitting Weather = Rainy

Evaluating features at depth1:
 IG(Temperature) = 0.5710
 IG(Soil Moisture) = 0.5710
 IG(Wind) = 0.9710
Best Feature == Wind
 Splitting Wind = Strong
 Splitting Wind = Weak
 Splitting Weather = Sunny

Evaluating features at depth1:
 IG(Temperature) = 0.9710
 IG(Soil Moisture) = 0.9710
 IG(Wind) = 0.5710
Best Feature == Temperature
 Splitting Temperature = Cool
 Splitting Temperature = Hot
 Splitting Temperature = Mild

Evaluating features at depth2:
 IG(Soil Moisture) = 1.0000
 IG(Wind) = 1.0000
Best Feature == Soil Moisture
 Splitting Soil Moisture = Dry
 Splitting Soil Moisture = Moist

 Final Decision Tree:

Weather=Cloudy:
 ---> Yes
Weather=Rainy:
 Wind=Strong:
  ---> No
 Wind=Weak:
  ---> Yes
Weather=Sunny:
 Temperature=Cool:
  ---> Yes
 Temperature=Hot:
  ---> No
 Temperature=